In [23]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
openai_api_key = os.getenv("OPENAI_API_KEY")

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

#### Buffer Memory
* RunnableWithMessageHistory keeps chat messages in an in-memory buffer for each session and passes them into the prompt.
* This replaces the legacy LLMChain and ConversationBufferMemory APIs, whose imports fail in this project's Python 3.14 environment.
* Use conversation.invoke({"question": "Bonjour !"}, config=config).content. Reuse the same session_id to retain history; choose a new one to start a separate conversation.
* History is not persisted: restarting the kernel or rerunning the setup cell clears it.

In [25]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder, 
    SystemMessagePromptTemplate, 
    HumanMessagePromptTemplate
)
from langchain_core.runnables.history import RunnableWithMessageHistory


prompt = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template("You are a nice chatbot having a friendly conversation with the user."),
        # The 'variable_name' should match the memory key used to store the chat history.
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

histories: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in histories:
        histories[session_id] = InMemoryChatMessageHistory()
    return histories[session_id]

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

config = {"configurable": {"session_id": "demo"}}
conversation.invoke({"question": "Bonjour !"}, config=config).content


c:\Users\ange0\Documents\Formations\GenAI\LangChain Latest\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


"Bonjour ! Comment ça va aujourd'hui ?"

In [26]:
conversation.invoke({"question": "Je m'appelle Ange et j'ai bougé 33 fois cette semaine."}, config=config).content

"Bonjour Ange ! Oh wow, cela fait beaucoup de mouvements en une semaine ! Tu sembles être une personne très active. Qu'as-tu fait de spécial pour te déplacer autant ?"

In [27]:
conversation.invoke({"question": "Si ma moyenne de mouvement de distance était de 10 km, combien de distance aurais-je parcourue cette semaine ?"}, config=config).content

"En parcourant 10 km à chacun de tes 33 mouvements, tu aurais parcouru un total de 330 km cette semaine. C'est une sacrée distance ! Tu as une grande capacité à explorer et à bouger, c'est impressionnant !"

In [28]:
conversation.invoke({"question": "Je m'appelle Ange et j'ai bougé 33 fois cette semaine."}, config=config).content

"Il semble y avoir une répétition. Est-ce que tu souhaites discuter d'autre chose ou as-tu des questions sur autre chose ? Je suis là pour t'aider !"


#### Conversation Buffer Window Memory
* This LCEL version trims the context sent to the model to the last 3 complete question/answer exchanges.
* It avoids the legacy `langchain.memory` and `ConversationChain` imports, which fail in this project's Python 3.14 environment.

In [29]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough

window_size = 3
window_prompt = RunnablePassthrough.assign(
    chat_history=lambda inputs: trim_messages(
        inputs["chat_history"],
        max_tokens=2 * window_size,
        token_counter=len,
        strategy="last",
        start_on="human",
        end_on=("human", "ai"),
    )
) | prompt

In [30]:
conversation_window = RunnableWithMessageHistory(
    window_prompt | llm,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

c:\Users\ange0\Documents\Formations\GenAI\LangChain Latest\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [31]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"De nos derniers échanges, je retiens que tu t'appelles Ange, que tu sembles très actif en bougeant beaucoup et que tu as fait beaucoup de déplacements cette semaine. J'ai également noté ta curiosité pour les calculs de distance. N'hésite pas à partager davantage sur tes activités ou à poser d'autres questions si tu le souhaites !"

In [32]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"De nos échanges, je retiens que tu t'appelles Ange, que tu sembles être une personne active et curieuse, et que tu aimes discuter de sujets variés comme les déplacements et les calculs. J'apprécie notre conversation et suis là pour répondre à toutes tes questions ou discuter de tout ce qui t'intéresse."

In [33]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"De notre conversation, je retiens que tu t'appelles Ange, que tu as bougé beaucoup cette semaine et que tu aimes discuter de divers sujets. Je suis là pour continuer à échanger avec toi et répondre à tes questions ou discuter de ce que tu veux. N'hésite pas à partager tes pensées ou tes préoccupations, je suis là pour t'écouter !"

In [34]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"De nos derniers échanges, je retiens que tu t'appelles Ange, que tu es quelqu'un de dynamique qui bouge beaucoup et qui s'intéresse aux calculs de distance. J'apprécie nos conversations et j'aime discuter avec toi de différents sujets. N'hésite pas à me poser d'autres questions ou à partager ce qui te préoccupe, je suis là pour t'écouter !"

In [35]:
conversation_window.invoke(
    {"question": "Ma couleur préférée est le vert ?"},
    config=config,
).content

"C'est super de savoir que ta couleur préférée est le vert ! C'est une couleur très apaisante et rafraîchissante. Est-ce que tu aimes le vert pour une raison particulière ? Ou c'est juste une couleur qui te plaît naturellement ?"

In [36]:
conversation_window.invoke(
    {"question": "Mon animal favori est le chat ?"},
    config=config,
).content

"Les chats sont des animaux adorables et tellement mignons ! C'est génial de savoir que ton animal favori est le chat. Ils sont souvent décrits comme indépendants, câlins et mystérieux à la fois. Est-ce que tu as un chat chez toi ou est-ce juste une affection naturelle pour ces félins ?"

In [37]:
conversation_window.invoke(
    {"question": "Je conduis une Opel Astra"},
    config=config,
).content

"C'est chouette de savoir que tu conduis une Opel Astra ! C'est une voiture réputée pour son confort et sa fiabilité. Est-ce que tu apprécies particulièrement conduire cette voiture ? Ou y a-t-il une autre raison pour laquelle tu as choisi ce modèle en particulier ? Qu'est-ce que tu aimes le plus quand tu conduis ta Opel Astra ?"

In [38]:
conversation_window.invoke(
    {"question": "Ma ville préférée est Paris"},
    config=config,
).content

"Paris est une ville magnifique ! Avec ses monuments emblématiques, ses délicieuses pâtisseries et son ambiance romantique, il n'est pas étonnant que ce soit une des villes préférées de beaucoup de gens. Est-ce que tu as une anecdote ou un endroit préféré à Paris que tu aimerais partager ? Ou est-ce simplement l'atmosphère de la ville qui te plaît tant ?"

In [39]:
conversation_window.invoke(
    {"question": "Quelles est ma couleur préférée ?"},
    config=config,
).content

"Hmm, je ne connais pas ta couleur préférée car tu ne me l'as pas dit. Quelle est ta couleur préférée ? Dis-moi et on pourra en discuter ensemble !"

In [40]:
conversation_window.invoke(
    {"question": "Que retenez-vous de nos derniers échanges ?"},
    config=config,
).content

"Je retiens que tu conduis une Opel Astra, que ta ville préférée est Paris et que je ne connais pas encore ta couleur préférée ! J'apprécie échanger avec toi et en apprendre davantage sur tes goûts et tes préférences. N'hésite pas à partager plus d'informations si tu le souhaites !"

In [41]:
conversation_window.invoke(
    {"question": "Quel est mon nom ?"},
    config=config,
).content

"Je suis désolé, mais je n'ai pas cette information. Cependant, je suis là pour discuter et échanger avec toi, peu importe ton nom ! Y a-t-il quelque chose en particulier que tu aimerais discuter ou savoir ?"

#### Chat Message History
* Save list of chat messages and then fetch them all.

In [42]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [43]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("hi!")

history.add_ai_message("whats up?")

In [44]:
history.messages

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='whats up?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]